# TP2: Pipeline Datalakehouse para API de clima
Data Engineering - CEL UTN

Modulo 2: Procesamiento de datos

Fecha limite de entrega: Domingo, 9 de Noviembre de 2025, 23:59

Alumno: Matias Falconaro

## Oportunidades de mejora aplicadas al TP1
Mejoras propuestas en TP1 sobre `Almacenamiento de datos`

Mejoras en TP1 sobre `Extracción full e incremental`

## Repositorio GitHub
[Pipeline Python artifact](https://github.com/matiasfalconaro/data-engineering-pipeline)

## Definición del Alcance para el dominio de datos

Criterio de Selección Poblacional: 5 ciudades argentinas más pobladas.

[Ciudades más pobladas de Argentina - Wikipedia](https://en.wikipedia.org/wiki/List_of_cities_in_Argentina_by_population)

Cobertura estratégica de los centros urbanos con mayor densidad poblacional para pronósticos climáticos que impacten a la mayor cantidad de habitantes.

## Arquitectura

![Arquitectura del Pipeline](https://drive.google.com/uc?export=view&id=1rxQImMYwympOK95-Y5dKXiYBXTpOo_N6)

## Decisiones de desarrollo

| Categoría | Modulo | Decisión | Justificación |
|-----------|--------|----------|---------------|
|**Investigacion APIs**| TP1 |Sub-foros de data engineering|[StackOverflow](https://stackoverflow.com/questions/29913271/weather-api-for-providing-weather-forecast-based-upon-location)<br> [Reddit](https://www.reddit.com/r/dataengineering/comments/14lcyxr/resources_for_weathergeospatial_data/)<br> [Medium](https://medium.com/@ajeet214/9-free-weather-apis-for-ai-data-projects-6bfc66022e46)|
| **API** | TP1 | OpenWeatherMap | Mencion recurrente en diferentes foros<br> Plan gratuito disponible<br> Documentación detallada<br> Tiempo de actividad confiable<br> Endpoints claros para clima actual y metadatos |
| **Extracción** | TP1 | **Incremental** (datos temporales)<br>**Full** (datos estáticos) | Clima actual se actualiza cada 10 minutos<br> Append para preservar histórico<br> Metadatos cambian raramente |
| **Checkpoint Incremental** | TP1 | Delta Lake Transaction Log | Elimina dependencia de archivos externos<br>Aprovecha metadata nativa de Delta<br>Más robusto que timestamps manuales |
| **Particionamiento** | TP1 | **Por fecha/hora** (temporales)<br>**Sin particionamiento** (estáticos) | Optimiza consultas temporales<br> Organiza grandes volúmenes de datos<br> Dataset estático es pequeño |
| **Verificación de Infraestructura** | TP1 | boto3 para validación de bucket MinIO | Valida conectividad S3 antes de ejecutar pipeline<br> Detecta errores de configuración tempranamente<br> Evita fallos silenciosos durante escritura Delta Lake<br> Compatibilidad estándar con cualquier S3-compatible storage<br> Permite realizar verificacion de forma programatica sobre el bucket<br>[StackOverflow](https://stackoverflow.com/questions/51104230/how-to-automate-permissions-for-aws-s3-bucket-objects)|
| **Elecciones Técnicas** | TP1 | Zona horaria UTC<br> Estructura modular | Evita problemas de zonas horarias<br> Código reutilizable y mantenible |
| **Manejo de Errores** | TP1 | Validación de respuestas API<br> Sistema de logging<br> Manejo datos faltantes<br> Verificación de directorios | Depuración y trazabilidad |
| **Performance** | TP1 | Extracción paralelizable<br> Particionamiento temporal<br> Procesamiento por lotes<br> Reintentos automáticos | Optimización consultas<br> Manejo eficiente de memoria<br> Resiliencia a fallos de red |
| **Documentación** | TP1 | Type hints<br> Docstring minimizados| Reduce la cantidad de lineas de código<br> Mejora la legibilidad de las funciones<br> Evita explicaciones sobre parametros y salidas en forma de string dentro del docstring|
| **Versionamiento Delta Lake** | TP1 | Archivos históricos preservados | MERGE crea nuevos archivos en lugar de modificar existentes<br> Archivos antiguos marcados como "removed" en transaction log<br> Permite time travel y auditoría de cambios<br> No indica duplicación de datos<br>|
| **Diseño de función de guardado** | TP1 | Función genérica `save_to_delta_lake()`<br> 3 modos: `append`, `overwrite`, `merge` | Evita duplicación de código entre datos temporales y estáticos<br> Código validado (0 duplicados, constraints activos)<br>|
| **Procesamiento** | TP2 | **Pipeline 4-etapas**:<br> →Limpieza <br> →Enriquecimiento Temporal<br> →Categorización Climática<br> →Agregación Diaria | Preserva datos crudos + genera datos enriquecidos<br> Transformaciones específicas para análisis climático<br> Optimizado para reporting y dashboards<br> Mantiene trazabilidad completa del procesamiento |
| **Transformaciones de Datos** | TP2 | **Específicas para contexto argentino**:<br> Rangos temperatura -20°C a +50°C<br> Estaciones hemisferio sur<br> Categorías percepción local<br> Agregados para toma de decisiones urbanas | Datos técnicos → información estratégica<br> Adaptado a realidad geográfica argentina<br> Optimizado para ciudades pobladas (~35% población nacional)<br> Habilita planificación de servicios públicos y alertas tempranas |

## [1] Dependencias

In [ ]:
!pip install boto3 deltalake pandas pyarrow requests treelib

In [ ]:
import boto3
import configparser
import json
import logging
import os
import pandas as pd
import pyarrow as pa
import requests
import time

from botocore.exceptions import ClientError
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError
from datetime import datetime, timezone
from pathlib import Path
from treelib import Tree
from typing import Dict, List, Optional, Union

## [2] Configuraciones del pipeline

In [ ]:
def load_config(config_file="/content/pipeline.conf"):
    """Carga configuración desde el archivo pipeline.conf"""
    if not Path(config_file).exists():
        raise FileNotFoundError(f"Configuration file '{config_file}' not found")

    config = configparser.ConfigParser()
    config.read(config_file)

    minio = {k: config.get('minio', k) for k in
             ['endpoint_url', 'access_key', 'secret_key', 'region', 'bucket_name']}

    base = f"s3://{minio['bucket_name']}/{config.get('data_lake', 'base_path')}"

    return {
        'api_key': config.get('api', 'api_key'),
        'base_url': config.get('api', 'base_url'),
        'api_timeout': config.getint('api', 'api_timeout'),
        'max_retries': config.getint('api', 'max_retries'),
        'cities': [c.strip() for c in config.get('api', 'cities').split(',')],
        'minio_config': minio,
        'storage_options': {
            "AWS_ENDPOINT_URL": minio["endpoint_url"],
            "AWS_ACCESS_KEY_ID": minio["access_key"],
            "AWS_SECRET_ACCESS_KEY": minio["secret_key"],
            "AWS_REGION": minio["region"],
            "AWS_ALLOW_HTTP": "true"
        },
        'data_lake_base': base,
        'temporal_data_path': f"{base}/{config.get('data_lake', 'temporal_path')}",
        'static_data_path': f"{base}/{config.get('data_lake', 'static_path')}",
        'processed_data_path': f"{base}/{config.get('data_lake', 'processed_path', fallback='processed_data')}",
        'processed_detailed_path': f"{base}/{config.get('data_lake', 'processed_detailed_path', fallback='processed_detailed')}"
    }

## [3] Gestion de logs

In [ ]:
def setup_logging() -> logging.Logger:
    """Configura el sistema de logging para el pipeline."""

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S',
        force=True
    )

    logger = logging.getLogger(__name__)
    logger.info("Pipeline de datos climáticos iniciado")

    return logger

## [4] Funciones de API client

In [ ]:
def make_api_request(endpoint: str, params: Dict) -> Optional[Dict]:
    """
    Realiza peticiones a la API con manejo de errores y reintentos automáticos.
    """
    for attempt in range(config['max_retries']):
        try:
            response = requests.get(endpoint, params=params, timeout=config['api_timeout'])
            response.raise_for_status()
            return response.json()
        except requests.exceptions.Timeout:
            if attempt == config['max_retries'] - 1:
                logger.error(f"Timeout al conectar con {endpoint} después de {config['max_retries']} intentos")
                return None
            time.sleep(2 ** attempt)
        except requests.exceptions.RequestException as e:
            if attempt == config['max_retries'] - 1:
                logger.error(f"Error en petición a {endpoint}: {e}")
                return None
            time.sleep(2 ** attempt)

    return None

In [ ]:
def get_current_weather(city: str) -> Optional[Dict]:
    """
    Extrae datos del clima actual para una ciudad específica.
    """
    endpoint = f"{config['base_url']}/weather"
    params = {
        "q": city,
        "appid": config['api_key'],
        "units": "metric",
        "lang": "es"
    }

    logger.info(f"Extrayendo datos climáticos para {city}")

    try:
        data = make_api_request(endpoint, params)

        if data and data.get("cod") == 200:
            # Agrego timestamp de extracción
            data['extraction_timestamp'] = datetime.now(timezone.utc).isoformat()
            data['extraction_city'] = city
            return data
        elif data:
            logger.warning(f"API retornó código {data.get('cod')} para {city}: {data.get('message', 'Sin mensaje')}")
            return None
        else:
            logger.warning(f"No se recibió respuesta para {city}")
            return None

    except Exception as e:
        logger.error(f"Error para {city}: {e}")
        return None


In [ ]:
def get_city_metadata(city_list: List[str]) -> List[Dict]:
    """
    Genera metadatos de la ciudad a partir de la respuesta de la API.
    Sirve como datos estaticos/referencia.
    """
    metadata = []

    for city in city_list:
        endpoint = f"{config['base_url']}/weather"
        params = {
            "q": city,
            "appid": config['api_key']
        }

        logger.info(f"Extrayendo metadatos para {city}")
        data = make_api_request(endpoint, params)

        if data:
            # Metadatos de ciudades
            metadata.append({
                "city_id": data.get("id"),
                "city_name": data.get("name"),
                "country": data.get("sys", {}).get("country"),
                "latitude": data.get("coord", {}).get("lat"),
                "longitude": data.get("coord", {}).get("lon"),
                "timezone_offset": data.get("timezone"),
                "last_updated": datetime.now(timezone.utc).isoformat()
            })

    return metadata

## [5] Funciones de transformacion de datos

In [ ]:
def weather_to_dataframe(weather_data: Union[Dict, List[Dict]]) -> pd.DataFrame:
    """
    Convierte respuestas crudas del clima de la API a DataFrame.
    """
    if isinstance(weather_data, dict):
        weather_data = [weather_data]

    records = []

    for data in weather_data:
        if data is None:
            continue

        # Tiempo de observación del clima (cuando se registró el dato)
        observation_dt = datetime.fromtimestamp(
            data.get("dt", 0), tz=timezone.utc
        )

        record = {
            # Identificadores
            "city_id": data.get("id"),
            "city_name": data.get("name"),
            "country": data.get("sys", {}).get("country"),

            # Condiciones climáticas
            "weather_main": data.get("weather", [{}])[0].get("main"),
            "weather_description": data.get("weather", [{}])[0].get("description"),

            # Datos de temperatura
            "temperature": data.get("main", {}).get("temp"),
            "feels_like": data.get("main", {}).get("feels_like"),
            "temp_min": data.get("main", {}).get("temp_min"),
            "temp_max": data.get("main", {}).get("temp_max"),

            # Otras métricas
            "pressure": data.get("main", {}).get("pressure"),
            "humidity": data.get("main", {}).get("humidity"),
            "visibility": data.get("visibility"),
            "wind_speed": data.get("wind", {}).get("speed"),
            "wind_direction": data.get("wind", {}).get("deg"),
            "cloudiness": data.get("clouds", {}).get("all"),

            # Marcas de tiempo
            "observation_time": datetime.fromtimestamp(
                data.get("dt", 0), tz=timezone.utc
            ).isoformat(),
            "extraction_timestamp": data.get("extraction_timestamp"),

            # Columnas de particionamiento
            "date": observation_dt.strftime("%Y-%m-%d"),
            "hour": observation_dt.strftime("%H")
        }

        records.append(record)

    df = pd.DataFrame(records)
    logger.info(f"DataFrame creado con {len(df)} registros")

    return df

In [ ]:
def metadata_to_dataframe(metadata: List[Dict]) -> pd.DataFrame:
    """
    Convierte los metadatos de la ciudad a Dataframe.
    """
    df = pd.DataFrame(metadata)
    logger.info(f"DataFrame de metadatos creado con {len(df)} registros")

    return df

## [6] Procesamiento de datos

In [ ]:
def clean_weather_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transformación 1: Limpieza de datos
    - Eliminar duplicados exactos
    - Manejar valores nulos
    - Validar rangos de datos
    """
    logger.info("Iniciando limpieza de datos climáticos")

    # Hacer copia para no modificar el original
    cleaned_df = df.copy()

    # 1. Eliminar duplicados exactos
    initial_count = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates()
    duplicates_removed = initial_count - len(cleaned_df)
    logger.info(f"Duplicados eliminados: {duplicates_removed}")

    # 2. Manejar valores nulos en columnas críticas
    critical_columns = ['temperature', 'humidity', 'pressure', 'wind_speed']
    for col in critical_columns:
        if col in cleaned_df.columns:
            null_count = cleaned_df[col].isnull().sum()
            if null_count > 0:
                # Para métricas numéricas, reemplazar con la mediana por ciudad
                cleaned_df[col] = cleaned_df.groupby('city_id')[col].transform(
                    lambda x: x.fillna(x.median())
                )
                logger.info(f"Valores nulos en {col}: {null_count} reemplazados")

    # 3. Validar rangos de datos
    # Temperatura en Argentina (rango razonable)
    temp_mask = (cleaned_df['temperature'] >= -20) & (cleaned_df['temperature'] <= 50)
    outliers_temp = len(cleaned_df) - temp_mask.sum()
    if outliers_temp > 0:
        logger.warning(f"Valores de temperatura fuera de rango: {outliers_temp}")
        cleaned_df = cleaned_df[temp_mask]

    # Humedad (0-100%)
    humidity_mask = (cleaned_df['humidity'] >= 0) & (cleaned_df['humidity'] <= 100)
    outliers_humidity = len(cleaned_df) - humidity_mask.sum()
    if outliers_humidity > 0:
        logger.warning(f"Valores de humedad fuera de rango: {outliers_humidity}")
        cleaned_df = cleaned_df[humidity_mask]

    logger.info(f"Limpieza completada. Registros finales: {len(cleaned_df)}")
    return cleaned_df

In [ ]:
def enrich_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transformación 2: Enriquecimiento temporal
    - Extraer día de semana
    - Flag de fin de semana
    - Categorías horarias
    - Estación del año
    """
    logger.info("Enriqueciendo características temporales")

    enriched_df = df.copy()

    # Convertir observation_time a datetime
    enriched_df['observation_datetime'] = pd.to_datetime(enriched_df['observation_time'])

    # Día de semana (0=Lunes, 6=Domingo)
    enriched_df['day_of_week'] = enriched_df['observation_datetime'].dt.dayofweek

    # Flag de fin de semana
    enriched_df['is_weekend'] = enriched_df['day_of_week'].isin([5, 6]).astype(int)

    enriched_df['time_category'] = enriched_df['hour'].apply(_get_time_category)

    enriched_df['month'] = enriched_df['observation_datetime'].dt.month
    enriched_df['season'] = enriched_df['month'].apply(_get_southern_season)

    logger.info("Enriquecimiento temporal completado")
    return enriched_df


def _get_time_category(hour):
  """ Categorías horarias """
  hour = int(hour)
  if 5 <= hour < 12:
      return 'mañana'
  elif 12 <= hour < 18:
      return 'tarde'
  elif 18 <= hour < 24:
      return 'noche'
  else:
      return 'madrugada'


def _get_southern_season(month):
  """ Estación del año (hemisferio sur) """
  if 12 <= month or month <= 2:
      return 'verano'
  elif 3 <= month <= 5:
      return 'otoño'
  elif 6 <= month <= 8:
      return 'invierno'
  else:
      return 'primavera'

In [ ]:
def create_weather_categories(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transformación 3: Categorización climática
    - Categorías de temperatura
    - Niveles de humedad
    - Intensidad de viento
    - Condiciones climáticas agrupadas
    """
    logger.info("Creando categorías climáticas")

    categorized_df = df.copy()

    categorized_df['temp_category'] = categorized_df['temperature'].apply(_categorize_temperature)

    categorized_df['humidity_category'] = categorized_df['humidity'].apply(_categorize_humidity)

    categorized_df['wind_intensity'] = categorized_df['wind_speed'].apply(_categorize_wind_speed)

    categorized_df['weather_summary'] = categorized_df.apply(
        lambda x: _summarize_weather_condition(x['weather_main'], x['weather_description']),
        axis=1
    )

    logger.info("Categorización climática completada")
    return categorized_df


def _categorize_temperature(temp):
  """ Categorías de temperatura (°C) """
  if temp < 10:
      return 'frío'
  elif 10 <= temp < 20:
      return 'templado'
  elif 20 <= temp < 30:
      return 'cálido'
  else:
      return 'caluroso'


def _categorize_humidity(humidity):
  """ Niveles de humedad """
  if humidity < 30:
        return 'seco'
  elif 30 <= humidity < 60:
        return 'confortable'
  elif 60 <= humidity < 80:
        return 'húmedo'
  else:
        return 'muy húmedo'


def _categorize_wind_speed(speed):
  """ Intensidad de viento (m/s) """
  if speed < 1:
      return 'calma'
  elif 1 <= speed < 5:
      return 'brisa leve'
  elif 5 <= speed < 10:
      return 'brisa moderada'
  elif 10 <= speed < 15:
        return 'ventoso'
  else:
       return 'muy ventoso'


def _summarize_weather_condition(main, description):
  """ Condiciones climáticas agrupadas """
  clear_conditions = ['clear', 'cielo claro']
  cloudy_conditions = ['clouds', 'nubes', 'few clouds', 'scattered clouds']
  rainy_conditions = ['rain', 'drizzle', 'lluvia', 'llovizna']

  if any(cond in str(main).lower() or cond in str(description).lower() for cond in clear_conditions):
      return 'despejado'
  elif any(cond in str(main).lower() or cond in str(description).lower() for cond in cloudy_conditions):
      return 'nublado'
  elif any(cond in str(main).lower() or cond in str(description).lower() for cond in rainy_conditions):
      return 'lluvioso'
  else:
      return 'otros'

In [ ]:
def create_city_daily_aggregates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transformación 4: Agregaciones diarias por ciudad
    - Métricas resumidas por día
    - Condición predominante
    - Variación térmica
    """
    logger.info("Creando agregados diarios por ciudad")

    # Asegurarse de que tenemos la columna date como datetime
    df['date_dt'] = pd.to_datetime(df['date'])

    # Agrupación por ciudad y fecha
    daily_agg = df.groupby(['city_id', 'city_name', 'date_dt']).agg({
        'temperature': ['mean', 'max', 'min', 'std'],
        'humidity': 'mean',
        'pressure': 'mean',
        'wind_speed': 'mean',
        'weather_summary': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'desconocido',
        'temp_category': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'desconocido'
    }).reset_index()

    # Aplanar columnas multi-index
    daily_agg.columns = [
        'city_id', 'city_name', 'date',
        'avg_temperature', 'max_temperature', 'min_temperature', 'temp_std',
        'avg_humidity', 'avg_pressure', 'avg_wind_speed',
        'predominant_weather', 'predominant_temp_category'
    ]

    # Calcular variación térmica diaria
    daily_agg['daily_temp_range'] = daily_agg['max_temperature'] - daily_agg['min_temperature']

    # Redondear valores decimales
    numeric_cols = ['avg_temperature', 'max_temperature', 'min_temperature', 'temp_std',
                   'avg_humidity', 'avg_pressure', 'avg_wind_speed', 'daily_temp_range']

    for col in numeric_cols:
        daily_agg[col] = daily_agg[col].round(2)

    # Agregar columnas de particionamiento
    daily_agg['year'] = daily_agg['date'].dt.year
    daily_agg['month'] = daily_agg['date'].dt.month
    daily_agg['day'] = daily_agg['date'].dt.day

    logger.info(f"Agregados diarios creados: {len(daily_agg)} registros")
    return daily_agg

In [ ]:
def process_weather_data() -> bool:
    """
    Orquesta todo el pipeline de procesamiento
    Aplica las transformaciones al DataFrame en cascada
    - Limpieza
    - Enriquecimiento temporal
    - Categorización
    - Agregados diarios
    """
    try:
        logger.info("=== INICIANDO PIPELINE DE PROCESAMIENTO ===")

        # 1. Datos crudos (TP1)
        logger.info("Leyendo datos climáticos crudos...")
        raw_weather_df = DeltaTable(
            config['temporal_data_path'],
            storage_options=config['storage_options']
        ).to_pandas()

        logger.info(f"Datos crudos cargados: {len(raw_weather_df)} registros")

        # 2. Transformaciones
        logger.info("Aplicando transformaciones...")

        cleaned_df = clean_weather_data(raw_weather_df)

        enriched_df = enrich_temporal_features(cleaned_df)

        categorized_df = create_weather_categories(enriched_df)

        daily_aggregates_df = create_city_daily_aggregates(categorized_df)

        # 3. Guardar datos procesados
        logger.info("Guardando datos procesados...")
        processed_path = config.get('processed_data_path', f"{config['data_lake_base']}/processed_data")

        save_to_delta_lake(
            df=daily_aggregates_df,
            base_path=processed_path,
            partition_cols=["year", "month"],
            mode="overwrite"
        )

        logger.info("=== PIPELINE DE PROCESAMIENTO COMPLETADO ===")
        logger.info(f"Datos procesados guardados en: {processed_path}")

        return True

    except Exception as e:
        logger.error(f"Error en el pipeline de procesamiento: {e}")
        return False

## [6] Configuracion del bucket de alamacenamiento

In [ ]:
def test_minio_connection():
    """
    Prueba la conexión a MinIO
    """
    logger.info("Probando conexión a MinIO")

    try:
        logger.info(f"Conectando a: {config['minio_config']['endpoint_url']}")

        s3_client = boto3.client(
            's3',
            endpoint_url=config['minio_config']["endpoint_url"],
            aws_access_key_id=config['minio_config']["access_key"],
            aws_secret_access_key=config['minio_config']["secret_key"],
            region_name=config['minio_config']["region"]
        )

        # Listo los buckets
        response = s3_client.list_buckets()
        buckets = [b['Name'] for b in response['Buckets']]

        logger.info("Conexión a MinIO exitosa")
        logger.info(f"Número de buckets: {len(buckets)}")
        logger.info(f"Buckets disponibles: {buckets}")
        return True

    except Exception as e:
        logger.error(f"Error conectando a MinIO: {e}")
        logger.error("Verificar: URL, credenciales, y que MinIO no este caido")
        return False

## [7] Funcion de almacenamiento - Data Lake




In [ ]:
def save_to_delta_lake(df: pd.DataFrame,
                      base_path: str,
                      partition_cols: Optional[List[str]] = None,
                      mode: str = "append",
                      merge_predicate: Optional[str] = None) -> None:
    """
    Guarda un DataFrame en formato Delta Lake en MinIO/S3.

    Args:
        df: DataFrame a guardar
        base_path: Ruta en S3/MinIO donde se guardará la tabla
        partition_cols: Columnas por las cuales particionar (opcional)
        mode: Modo de escritura - 'append', 'overwrite' o 'merge' (default: 'append')
        merge_predicate: Condición SQL para merge (requerido solo si mode='merge')

    Modos soportados:
        - 'append': Agrega nuevos registros sin verificar duplicados
        - 'overwrite': Reemplaza todos los datos existentes
        - 'merge': Actualiza registros existentes e inserta nuevos (upsert)

    NOTA: Las constraints se aplican solo durante la creación inicial de la tabla.
    """
    try:
        logger.info(f"Guardando {len(df)} registros en {base_path} (modo: {mode})")

        # Convierto a PyArrow
        pa_table = pa.Table.from_pandas(df)

        if mode == "merge":
            if not merge_predicate:
                raise ValueError("merge_predicate es requerido para mode='merge'")

            try:
                dt = DeltaTable(base_path, storage_options=config['storage_options'])
                dt.merge(
                    source=df,
                    predicate=merge_predicate,
                    source_alias="source",
                    target_alias="target"
                ).when_matched_update_all().when_not_matched_insert_all().execute()
                logger.info(f"MERGE exitoso: {len(df)} registros procesados")

            except TableNotFoundError:
                logger.info("Tabla no existe, creando inicialmente")
                write_deltalake(
                    base_path,
                    pa_table,
                    mode="overwrite",
                    partition_by=partition_cols,
                    storage_options=config['storage_options']
                )
                # Constraints solo en la creación
                dt = DeltaTable(base_path, storage_options=config['storage_options'])
                _apply_initial_constraints(dt, base_path)
                logger.info(f"Tabla creada con constraints: {len(df)} registros")

        elif mode in ["append", "overwrite"]:
            # Verificar existencia
            table_exists = True
            try:
                DeltaTable(base_path, storage_options=config['storage_options'])
            except TableNotFoundError:
                table_exists = False

            write_deltalake(
                base_path,
                pa_table,
                mode=mode,
                partition_by=partition_cols,
                storage_options=config['storage_options']
            )
            logger.info(f"{mode.upper()} exitoso: {len(df)} registros")

            # Constraints en 1ra ejecucion
            if not table_exists and mode == "overwrite":
                dt = DeltaTable(base_path, storage_options=config['storage_options'])
                _apply_initial_constraints(dt, base_path)
                logger.info("Constraints iniciales aplicadas")

        else:
            raise ValueError(f"Modo no soportado: {mode}. Usar: append, overwrite o merge")

    except Exception as e:
        logger.error(f"Error escribiendo en Delta Lake ({base_path}): {e}")
        raise


def _apply_initial_constraints(dt: DeltaTable, base_path: str) -> None:
    """
    Aplica constraints de integridad solo durante la creación inicial de la tabla.
    Esta función debe ejecutarse una sola vez por tabla.
    """
    constraints = {}

    if "temporal" in base_path:
        constraints = {
            "pk_not_null": "city_id IS NOT NULL AND date IS NOT NULL AND hour IS NOT NULL",
            "valid_humidity_range": "humidity >= 0 AND humidity <= 100"
        }

    elif "metadata" in base_path:
        constraints = {
            "city_id_not_null": "city_id IS NOT NULL",
            "valid_coordinates": "latitude BETWEEN -90 AND 90 AND longitude BETWEEN -180 AND 180"
        }

    # Aplicar cada constraint individualmente
    for constraint_name, constraint_expr in constraints.items():
        try:
            dt.alter.add_constraint({constraint_name: constraint_expr})
            logger.info(f"Constraint inicial '{constraint_name}' aplicado exitosamente")
        except Exception as e:
            logger.warning(f"Error aplicando constraint inicial '{constraint_name}': {e}")

## [8] Uso con datos temporales (Clima)

In [ ]:
"""
EXTRACCIÓN INCREMENTAL:
• Checkpoint: Delta Lake Transaction Log (nativo)
• Método: Lee último timestamp de commits desde _delta_log
• Ventajas:
  - No requiere archivos externos
  - Consistencia transaccional garantizada
  - Time-travel incorporado
• Frecuencia: 10 minutos (configurable)
"""

def extract_and_save_weather_data() -> bool:
    """
    Extracción INCREMENTAL usando Delta Lake transaction log.
    """
    logger.info("EXTRACCIÓN INCREMENTAL: Datos Climáticos")

    # Verificar cambios (usando transaction log)
    if not _should_extract_incremental(interval_minutes=10):
        return True

    # Extraer datos
    extraction_start = datetime.now(timezone.utc)
    weather_data = []

    for city in config['cities']:
        logger.info(f"Extrayendo: {city}")
        data = get_current_weather(city)
        if data:
            weather_data.append(data)

    if not weather_data:
        logger.warning("No se obtuvieron datos")
        return False

    df_weather = weather_to_dataframe(weather_data)
    logger.info(f"Datos extraídos: {len(df_weather)} registros")

    # MERGE con métricas
    logger.info("Ejecutando MERGE incremental...")
    metrics = save_to_delta_lake(
        df=df_weather,
        base_path=config['temporal_data_path'],
        partition_cols=["date", "hour"],
        mode="merge",
        merge_predicate="target.city_id = source.city_id AND target.date = source.date AND target.hour = source.hour"
    )

    # Mostrar métricas
    duration = (datetime.now(timezone.utc) - extraction_start).total_seconds()

    logger.info("MÉTRICAS DE EXTRACCIÓN INCREMENTAL:")
    logger.info(f"   • Nuevos insertados:      {metrics.get('inserted', 0)}")
    logger.info(f"   • Actualizados:           {metrics.get('updated', 0)}")
    logger.info(f"   • Total procesado:        {metrics.get('total_processed', 0)}")
    logger.info(f"   • Duración:               {duration:.2f}s")
    logger.info(f"   • Método checkpoint:      Delta Lake transaction log (nativo)")

    logger.info("Extracción incremental completada")
    return True


def _get_last_extraction_timestamp() -> Optional[datetime]:
    """Lee timestamp del Delta Lake transaction log"""
    try:
        dt = DeltaTable(
            config['temporal_data_path'],
            storage_options=config['storage_options']
        )
        history = dt.history(limit=1).to_pandas()

        if len(history) > 0:
            last_timestamp = history['timestamp'].iloc[0]
            if last_timestamp.tzinfo is None:
                last_timestamp = last_timestamp.replace(tzinfo=timezone.utc)
            return last_timestamp

    except TableNotFoundError:
        return None
    except Exception as e:
        logger.warning(f"Error leyendo Delta log: {e}")
        return None

    return None


def _should_extract_incremental(interval_minutes: int = 10) -> bool:
    """Valida si debe ejecutarse extracción usando metadatos de Delta Lake."""
    last_extraction = _get_last_extraction_timestamp()

    if last_extraction is None:
        logger.info("Primera ejecución - proceder con extracción")
        return True

    elapsed = datetime.now(timezone.utc) - last_extraction
    elapsed_minutes = elapsed.total_seconds() / 60

    if elapsed_minutes < interval_minutes:
        logger.info(f"Omitiendo extracción: última hace {elapsed_minutes:.1f} min")
        logger.info(f"Próxima extracción en: {interval_minutes - elapsed_minutes:.1f} min")
        return False

    logger.info(f"Proceder: última extracción hace {elapsed_minutes:.1f} min")
    return True

## [9] Uso con datos estaticos (Metadatos)

In [ ]:
def refresh_city_metadata() -> bool:
    """
    Extracción FULL de metadatos de ciudades.

    Estrategia: Overwrite completo porque los metadatos
    cambian raramente y el dataset es pequeño.
    """
    metadata = get_city_metadata(config['cities'])

    if metadata:
        df_metadata = metadata_to_dataframe(metadata)

        # EXTRACCIÓN FULL
        save_to_delta_lake(
            df=df_metadata,
            base_path=config['static_data_path'],
            mode="overwrite"
        )
        logger.info(f"Extracción FULL completada: {len(df_metadata)} ciudades")
        return True

    logger.warning("No se obtuvieron metadatos de ciudades")
    return False

## [10] Verificaciones

In [ ]:
def verify_delta_table(path, table_name):
    """
    Verificación de tablas Delta
    """
    logger.info(f"Verificando: {table_name}")

    try:
        dt = DeltaTable(path, storage_options=config['storage_options'])
        df = dt.to_pandas()

        logger.info(f"CONEXIÓN EXITOSA - Registros: {len(df):,}\n")

        # Info general
        df.info()
        print()

        # Datos
        print("PRIMERAS 5 FILAS:")
        display(df.head(5))
        print("\n")

        # Estadísticas
        numeric_cols = df.select_dtypes(include=['number']).columns
        if len(numeric_cols) > 0:
            print("ESTADÍSTICAS:")
            display(df[numeric_cols].describe())
            print()

        # Valido duplicados
        if 'city_id' in df.columns:
            # Datos temporales crudos
            if 'date' in df.columns and 'hour' in df.columns:
                key_cols = ['city_id', 'date', 'hour']
            # Datos procesados (agregados diarios)
            elif 'date' in df.columns:
                key_cols = ['city_id', 'date']
            # Metadatos
            else:
                key_cols = ['city_id']

            duplicados = df.duplicated(subset=key_cols).sum()
            print(f"Duplicados: {duplicados}")
            print()

        return True

    except Exception as e:
        logger.error(f"Error verificando {table_name}: {e}")
        return False

In [ ]:
def show_bucket_tree():
    """Muestra árbol completo del data lake"""

    print("ESTRUCTURA DEL DATA LAKE")
    print()

    s3 = boto3.client('s3',
                      endpoint_url=config['minio_config']["endpoint_url"],
                      aws_access_key_id=config['minio_config']["access_key"],
                      aws_secret_access_key=config['minio_config']["secret_key"])

    bucket = config['minio_config']["bucket_name"]

    # Creo árbol
    tree = Tree()
    tree.create_node(bucket, bucket)

    # Proceso objetos
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket):
        for obj in page.get('Contents', []):
            parts = obj['Key'].split('/')
            parent = bucket

            for i, part in enumerate(parts):
                node_id = '/'.join(parts[:i+1])

                if not tree.contains(node_id):
                    tree.create_node(part, node_id, parent=parent)

                parent = node_id

    tree.show()

## [11] Orquestacion del pipeline

In [ ]:
# Configuración global
config = load_config()
logger = setup_logging()

15:12:50 - INFO - Pipeline de datos climáticos iniciado


In [ ]:
def main(enable_extraction: bool = True,
        enable_processing: bool = True,
        enable_verification: bool = True,
        enable_metadata_refresh: bool = True) -> bool:
    """
    Función principal que orquesta el pipeline completo de datos climáticos.
    """
    try:
        logger.info("INICIANDO PIPELINE DE DATOS CLIMÁTICOS")

        if not test_minio_connection():
            logger.error("Falló la conexión a MinIO")
            return False

        # EXTRACCIÓN DE DATOS
        if enable_extraction:
            logger.info("=== EXTRACCIÓN DE DATOS CLIMÁTICOS ===")
            if not extract_and_save_weather_data():
                logger.error("Falló la extracción de datos climáticos")
                return False

        # METADATOS DE CIUDADES
        if enable_metadata_refresh:
            logger.info("=== ACTUALIZACIÓN DE METADATOS ===")
            if not refresh_city_metadata():
                logger.error("Falló la actualización de metadatos")
                return False

        # PROCESAMIENTO TP2
        if enable_processing:
            logger.info("=== TRANSFORMACIÓN Y ENRIQUECIMIENTO (TP2) ===")
            if not process_weather_data():
                logger.error("Falló el procesamiento TP2")
                return False

        # VERIFICACIONES
        if enable_verification:
            logger.info("=== VERIFICACIÓN DE DATOS ===")

            verification_results = []

            # Datos climáticos
            verification_results.append(
                verify_delta_table(config['temporal_data_path'], "Datos Climáticos (Crudos)")
            )
            print("-" * 60 + "\n")

            # Metadatos
            verification_results.append(
                verify_delta_table(config['static_data_path'], "Metadatos de Ciudades")
            )
            print("-" * 60 + "\n")

            # Datos procesados si existen
            processed_path = config.get('processed_data_path')
            if processed_path and enable_processing:
                verification_results.append(
                    verify_delta_table(processed_path, "Datos Procesados (Agregados Diarios)")
                )
                print("-" * 60 + "\n")

            # Mostrar estructura del data lake
            show_bucket_tree()

            # Resumen
            success_count = sum(verification_results)
            total_checks = len(verification_results)
            logger.info(f"Verificaciones: {success_count}/{total_checks} exitosas")

        logger.info("PIPELINE COMPLETADO EXITOSAMENTE")
        return True

    except Exception as e:
        logger.error(f"ERROR CRÍTICO: {e}")
        return False

In [ ]:
# EJECUTO EL PIPELINE
if __name__ == "__main__":
    print("INICIANDO EJECUCIÓN DEL PIPELINE...")
    success = main(enable_extraction=False,
              enable_processing=True,
              enable_verification=True,
              enable_metadata_refresh=False)

    if success:
        print("PIPELINE COMPLETADO EXITOSAMENTE")
    else:
        print("PIPELINE FALLÓ")

15:25:44 - INFO - INICIANDO PIPELINE DE DATOS CLIMÁTICOS
15:25:44 - INFO - Probando conexión a MinIO
15:25:44 - INFO - Conectando a: http://31.97.241.212:9000


INICIANDO EJECUCIÓN DEL PIPELINE...


15:25:45 - INFO - Conexión a MinIO exitosa
15:25:45 - INFO - Número de buckets: 1
15:25:45 - INFO - Buckets disponibles: ['matiasfalconaro-bucket']
15:25:45 - INFO - === TRANSFORMACIÓN Y ENRIQUECIMIENTO (TP2) ===
15:25:45 - INFO - === INICIANDO PIPELINE DE PROCESAMIENTO ===
15:25:45 - INFO - Leyendo datos climáticos crudos...
15:25:47 - INFO - Datos crudos cargados: 5 registros
15:25:47 - INFO - Aplicando transformaciones...
15:25:47 - INFO - Iniciando limpieza de datos climáticos
15:25:47 - INFO - Duplicados eliminados: 0
15:25:47 - INFO - Limpieza completada. Registros finales: 5
15:25:47 - INFO - Enriqueciendo características temporales
15:25:47 - INFO - Enriquecimiento temporal completado
15:25:47 - INFO - Creando categorías climáticas
15:25:47 - INFO - Categorización climática completada
15:25:47 - INFO - Creando agregados diarios por ciudad
15:25:47 - INFO - Agregados diarios creados: 5 registros
15:25:47 - INFO - Guardando datos procesados...
15:25:47 - INFO - Guardando 5 regist

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   city_id               5 non-null      int64  
 1   city_name             5 non-null      object 
 2   country               5 non-null      object 
 3   weather_main          5 non-null      object 
 4   weather_description   5 non-null      object 
 5   temperature           5 non-null      float64
 6   feels_like            5 non-null      float64
 7   temp_min              5 non-null      float64
 8   temp_max              5 non-null      float64
 9   pressure              5 non-null      int64  
 10  humidity              5 non-null      int64  
 11  visibility            5 non-null      int64  
 12  wind_speed            5 non-null      float64
 13  wind_direction        5 non-null      int64  
 14  cloudiness            5 non-null      int64  
 15  observation_time      5 non

,city_id,city_name,country,weather_main,weather_description,temperature,feels_like,temp_min,temp_max,pressure,humidity,visibility,wind_speed,wind_direction,cloudiness,observation_time,extraction_timestamp,date,hour
0,3435910,Buenos Aires,AR,Clouds,algo de nubes,29.93,30.23,28.91,32.13,1011,45,10000,5.06,15,12,2025-11-15T15:21:08+00:00,2025-11-15T15:24:05.920348+00:00,2025-11-15,15
1,3860259,Córdoba,AR,Clouds,nubes dispersas,29.47,30.43,27.88,29.47,1009,51,10000,8.23,20,40,2025-11-15T15:22:37+00:00,2025-11-15T15:24:05.935562+00:00,2025-11-15,15
2,3838583,Rosario,AR,Clear,cielo claro,31.41,35.73,30.95,31.73,1007,60,10000,6.17,340,0,2025-11-15T15:22:37+00:00,2025-11-15T15:24:05.949938+00:00,2025-11-15,15
3,3432043,La Plata,AR,Clouds,algo de nubes,29.49,28.50,29.49,29.49,1011,33,10000,5.58,333,15,2025-11-15T15:22:37+00:00,2025-11-15T15:24:05.969599+00:00,2025-11-15,15
4,3430863,Mar del Plata,AR,Clear,cielo claro,31.01,30.52,31.01,31.01,1008,37,10000,10.80,320,0,2025-11-15T15:22:37+00:00,2025-11-15T15:24:05.985176+00:00,2025-11-15,15


...



ESTADÍSTICAS:


,city_id,avg_temperature,max_temperature,min_temperature,temp_std,avg_humidity,avg_pressure,avg_wind_speed,daily_temp_range,year,month,day
count,5.000000e+00,5.000000,5.000000,5.000000,0.0,5.000000,5.000000,5.000000,5.0,5.0,5.0,5.0
mean,3.599532e+06,30.262000,30.262000,30.262000,NaN,45.200000,1009.200000,7.168000,0.0,2025.0,11.0,15.0
std,2.282531e+05,0.895946,0.895946,0.895946,NaN,10.825895,1.788854,2.360121,0.0,0.0,0.0,0.0
min,3.430863e+06,29.470000,29.470000,29.470000,NaN,33.000000,1007.000000,5.060000,0.0,2025.0,11.0,15.0
25%,3.432043e+06,29.490000,29.490000,29.490000,NaN,37.000000,1008.000000,5.580000,0.0,2025.0,11.0,15.0
50%,3.435910e+06,29.930000,29.930000,29.930000,NaN,45.000000,1009.000000,6.170000,0.0,2025.0,11.0,15.0
75%,3.838583e+06,31.010000,31.010000,31.010000,NaN,51.000000,1011.000000,8.230000,0.0,2025.0,11.0,15.0
max,3.860259e+06,31.410000,31.410000,31.410000,NaN,60.000000,1011.000000,10.800000,0.0,2025.0,11.0,15.0



Duplicados: 0

------------------------------------------------------------

ESTRUCTURA DEL DATA LAKE



15:26:03 - INFO - Verificaciones: 3/3 exitosas
15:26:03 - INFO - PIPELINE COMPLETADO EXITOSAMENTE


matiasfalconaro-bucket
└── data_lake
    ├── city_metadata
    │   ├── _delta_log
    │   │   ├── 00000000000000000000.json
    │   │   ├── 00000000000000000001.json
    │   │   └── 00000000000000000002.json
    │   └── part-00000-c8b0f928-1523-46b3-a602-b409b1dc05ac-c000.snappy.parquet
    ├── processed_data
    │   ├── _delta_log
    │   │   └── 00000000000000000000.json
    │   └── year=2025
    │       └── month=11
    │           └── part-00000-e59c0cc9-47a1-4a47-b569-65e4ced2f440-c000.snappy.parquet
    └── weather_temporal
        ├── _delta_log
        │   ├── 00000000000000000000.json
        │   ├── 00000000000000000001.json
        │   └── 00000000000000000002.json
        └── date=2025-11-15
            └── hour=15
                └── part-00000-4597033f-bd7f-491f-b130-8134271bb611-c000.snappy.parquet

PIPELINE COMPLETADO EXITOSAMENTE
